# Volume 12, Chapter 12: Elliptic curves (Python companion)

Exercises first, solutions at the end. We implement the group law with exact fractions and test associativity, compute torsion subgroups by Nagell–Lutz and meet Mazur's list, watch heights quadruple under doubling, hunt congruent numbers and test Tunnell's criterion, and finish with elliptic-curve cryptography done properly: **X25519**, the key exchange in TLS 1.3, SSH and Signal, checked against the official test vectors of RFC 7748. Requires `cypari2`.

In [ ]:
import math, random, time
from fractions import Fraction as Fr
from collections import Counter
import cypari2
pari = cypari2.Pari()
pari.allocatemem(2 * 10**9, silent=True)
random.seed(12)

## Exercises

**P12.1 (★) The group law.** Implement $P+Q$ on $y^2=x^3+ax+b$ over $\mathbb Q$ with `Fraction`, using formulas (12.1), with `None` for $O$. Check associativity (Theorem 12.2) on random combinations of the points $(12,36)$ and $(-3,9)$ of $y^2=x^3-36x$, including doublings, and check that three collinear points add up to $O$.

**P12.2 (★★) Torsion by Nagell–Lutz.** Compute $E(\mathbb Q)_{\rm tors}$ for all curves $y^2=x^3+ax+b$ with $|a|,|b|\le 40$ by Theorem 12.6: list the integral points with $y=0$ or $y^2\mid4a^3+27b^2$, and keep those of finite order (Proposition 12.6 gives a test). Compare with PARI's `elltors` and tally the groups that occur (Mazur's list allows fifteen).

**P12.3 (★★) Heights.** On $y^2=x^3-36x$ with $P=(12,36)$, compute $h(2^kP)=\log H(x(2^kP))$ for $k\le 8$ and the ratios $h(2^kP)/4^k$. Compare the limit with PARI's canonical height `ellheight` (check which normalisation it uses).

**P12.4 (★★) Congruent numbers.** For $n\le 100$ squarefree, compute the rank of $E_n:y^2=x^3-n^2x$ with `ellrank`, list the congruent ones with a triangle for each, and check that every squarefree $n\equiv5,6,7\pmod 8$ in range is congruent. Then test Tunnell's criterion for squarefree $n<1000$: for odd $n$, count $A=\#\{n=2x^2+y^2+32z^2\}$ and $B=\#\{n=2x^2+y^2+8z^2\}$ and compare "$B=2A$" with "rank $>0$" (for even $n$, use $n/2=4x^2+y^2+32z^2$ and $n/2=4x^2+y^2+8z^2$).

**P12.5 (★★★) CS application: X25519.** Curve25519 is $v^2=u^3+486662u^2+u$ over $\mathbb F_p$, $p=2^{255}-19$. Implement the X25519 function of RFC 7748 (scalar clamping and the *Montgomery ladder* on the $u$-coordinate, with a conditional swap at each bit), and check it against the RFC's test vectors and its Diffie–Hellman example. Using PARI, count the points of the curve and of its quadratic twist, and explain why X25519 does not need to check that the received $u$ is on the curve.

## Solutions

### P12.1
**How to find it.** Formulas (12.1) with exact rationals; $O$ is represented by `None`. Three points are collinear iff the determinant of their projective coordinates vanishes.

In [ ]:
def add(P, Q, a):
    if P is None: return Q
    if Q is None: return P
    (x1, y1), (x2, y2) = P, Q
    if x1 == x2 and y1 == -y2:
        return None
    lam = (3 * x1 * x1 + a) / (2 * y1) if P == Q else (y2 - y1) / (x2 - x1)
    x3 = lam * lam - x1 - x2
    return (x3, -y1 - lam * (x3 - x1))

def mul(k, P, a):
    R = None
    if k < 0:
        k, P = -k, (P[0], -P[1])
    while k:
        if k & 1: R = add(R, P, a)
        P = add(P, P, a); k >>= 1
    return R

a, b = Fr(-36), Fr(0)
P0, Q0 = (Fr(12), Fr(36)), (Fr(-3), Fr(9))
on = lambda P: P is None or P[1]**2 == P[0]**3 + a*P[0] + b
print("on the curve:", on(P0), on(Q0))
ok = True
for _ in range(30):
    P, Q, R = [add(mul(random.randint(-3, 3), P0, a), mul(random.randint(-3, 3), Q0, a), a) for _ in range(3)]
    ok &= add(add(P, Q, a), R, a) == add(P, add(Q, R, a), a)
    ok &= add(add(P, P, a), Q, a) == add(P, add(P, Q, a), a)
print("associativity on 30 random triples (with doublings):", ok)
P, Q = mul(2, P0, a), Q0
R = (add(P, Q, a)[0], -add(P, Q, a)[1])            # the third point on the line PQ
det = P[0]*(Q[1]-R[1]) - P[1]*(Q[0]-R[0]) + (Q[0]*R[1]-Q[1]*R[0])
print("P + Q + (third point) = O:", add(add(P, Q, a), R, a) is None, "; collinear:", det == 0)

### P12.2
**How to find it.** For each $y$ with $y=0$ or $y^2\mid D$, the integral roots of $x^3+ax+b-y^2$ are among the divisors of its constant term (or just use PARI's `polrootsreal` and round). To decide whether a candidate $P$ has finite order, pick a good prime $p$ and $N_p=\#\tilde E(\mathbb F_p)$: by Proposition 12.6, $|E(\mathbb Q)_{\rm tors}|$ divides $N_p$, so $P$ is torsion iff $N_pP=O$.

In [ ]:
def nl_torsion(a, b):
    D = 4*a**3 + 27*b**2
    ys = [0] + [y for y in range(1, math.isqrt(abs(D)) + 1) if D % (y*y) == 0]
    pts = []
    for y in ys:
        for r in pari.polrootsreal(pari(f"x^3+({a})*x+({b})-{y*y}")):
            x = round(float(r))
            if x**3 + a*x + b == y*y:
                pts += [(Fr(x), Fr(y))] + ([(Fr(x), Fr(-y))] if y else [])
    p = next(q for q in pari.primes(30) if (2 * D) % q)            # a good prime
    Np = int(pari.ellcard(pari.ellinit([a, b], p)))
    return [None] + [P for P in set(pts) if mul(Np, P, Fr(a)) is None]

def group_name(tors, a):
    n = len(tors)
    two = sum(1 for P in tors if P is not None and P[1] == 0)
    return f"Z/2 x Z/{n//2}" if two == 3 else f"Z/{n}"

tally, mismatch = Counter(), 0
t0 = time.time()
for a in range(-40, 41):
    for b in range(-40, 41):
        if 4*a**3 + 27*b**2 == 0:
            continue
        T = nl_torsion(a, b)
        ref = pari.elltors(pari.ellinit([a, b]))
        mismatch += len(T) != int(ref[0])
        tally[group_name(T, a)] += 1
print(f"{sum(tally.values())} curves, {mismatch} disagreements with elltors, {time.time()-t0:.1f}s")
print(sorted(tally.items(), key=lambda t: (len(t[0]), t[0])))

Small coefficients already produce six of Mazur's fifteen groups; the others ($\mathbb Z/5$, $\mathbb Z/7$, $\mathbb Z/8$, $\mathbb Z/9$, $\mathbb Z/10$, $\mathbb Z/12$, $\mathbb Z/2\times\mathbb Z/4$, $\mathbb Z/2\times\mathbb Z/6$, $\mathbb Z/2\times\mathbb Z/8$) need larger coefficients in this short Weierstrass form, like $y^2=x^3-43x+166$ for $\mathbb Z/7$ (Exercise 12.6).

### P12.3
**How to find it.** Doubling with exact fractions; $H(x)=\max(|{\rm num}|,|{\rm den}|)$.

In [ ]:
P = (Fr(12), Fr(36)); a = Fr(-36)
for k in range(9):
    x = P[0]
    h = math.log(max(abs(x.numerator), x.denominator))
    digits = int(x.denominator.bit_length() * math.log10(2)) + 1
    print(f"k={k}: about {digits:5d} digits in the denominator of x;   h/4^k = {h/4**k:.6f}")
    P = add(P, P, a)
E6 = pari.ellinit([-36, 0])
print("PARI ellheight(P) =", float(pari.ellheight(E6, [12, 36])))

The ratios converge fast to $\hat h(P)\approx0.8886$: the errors are bounded by Lemmas 12.11 and 12.12 and get divided by $4$ at each step. The digits quadruple at each doubling, which is why Figure 12.2 stops at $4P$. PARI's `ellheight` returns the same number; beware that some books (Silverman's, for instance) define the canonical height as half of it, with $h(P)=\frac12\log H(x)$. Always check the normalisation when comparing heights from different sources.

### P12.4
**How to find it.** `ellrank(E)` returns lower and upper bounds for the rank (from 2-descent, the method of Section 12.3) and points; for these small $n$ the bounds agree. A point $(x,y)$ gives the triangle of Proposition 12.13. For Tunnell's counts, loop over $|z|\le\sqrt{n/32}$ (or $\sqrt{n/8}$), $|x|\le\sqrt{n/2}$, and test whether the rest is a square.

In [ ]:
def squarefree(n): return all(n % (q*q) for q in range(2, math.isqrt(n) + 1))
cong, firsts = [], {}
for n in range(1, 101):
    if not squarefree(n): continue
    E = pari.ellinit([-n*n, 0])
    lo, hi, _, pts = pari.ellrank(E)
    assert lo == hi, n
    if lo > 0:
        cong.append(n)
        x, y = pts[0][0], pts[0][1]
        A, B, Cc = abs((x*x - n*n) / y), abs(2*n*x / y), abs((x*x + n*n) / y)
        if len(firsts) < 8: firsts[n] = (A, B, Cc)
print("congruent squarefree n <= 100:", cong)
for n, t in firsts.items():
    print(n, [str(s) for s in t])
print("squarefree n = 5,6,7 mod 8 that are not congruent:", [n for n in range(1, 101) if squarefree(n) and n % 8 in (5, 6, 7) and n not in cong])

In [ ]:
def count(n, c1, c2, c3):
    tot = 0
    for z in range(-math.isqrt(n // c3), math.isqrt(n // c3) + 1):
        for x in range(-math.isqrt(n // c1), math.isqrt(n // c1) + 1):
            r = n - c1*x*x - c3*z*z
            if r >= 0:
                s = math.isqrt(r)
                tot += (s*s == r) * (1 if s == 0 else 2)
    return tot
agree = disagree = 0
for n in range(1, 1000):
    if not squarefree(n): continue
    m, c1 = (n, 2) if n % 2 else (n // 2, 4)
    tunnell = count(m, c1, 1, 8) == 2 * count(m, c1, 1, 32)
    rank = int(pari.ellrank(pari.ellinit([-n*n, 0]))[0])
    if tunnell == (rank > 0): agree += 1
    else: disagree += 1
print(f"Tunnell's criterion vs rank > 0 for squarefree n < 1000: {agree} agree, {disagree} disagree")

No disagreement. One direction ("congruent $\Rightarrow B=2A$") is Tunnell's theorem; the converse is what BSD would add, and for each $n$ where the criterion holds, `ellrank` simply finds the point. The congruent numbers below $100$ include all squarefree $n\equiv5,6,7\pmod8$, as conjectured.

### P12.5
**How to find it.** RFC 7748, Section 5: clamp the scalar (clear the three low bits, so the scalar is a multiple of the cofactor $8$; clear bit $255$, set bit $254$), then run the ladder on $(x_2:z_2)=[k]P$ and $(x_3:z_3)=[k+1]P$ with the differential addition formulas, swapping the pair according to each bit. The ladder performs the same operations whatever the bits, which is what "constant time" means (Python's big integers are not constant time; real implementations use fixed-size limbs).

In [ ]:
p25519 = 2**255 - 19; A24 = 121665

def decode_u(b):
    b = bytearray(b); b[31] &= 127
    return int.from_bytes(b, 'little') % p25519

def decode_k(b):
    b = bytearray(b); b[0] &= 248; b[31] &= 127; b[31] |= 64
    return int.from_bytes(b, 'little')

def x25519(kb, ub):
    p = p25519
    k, x1 = decode_k(kb), decode_u(ub)
    x2, z2, x3, z3, swap = 1, 0, x1, 1, 0
    for t in reversed(range(255)):
        kt = (k >> t) & 1
        swap ^= kt
        if swap: x2, x3, z2, z3 = x3, x2, z3, z2       # conditional swap
        swap = kt
        A = (x2 + z2) % p; AA = A * A % p; B = (x2 - z2) % p; BB = B * B % p; E = (AA - BB) % p
        C = (x3 + z3) % p; D = (x3 - z3) % p; DA = D * A % p; CB = C * B % p
        x3 = (DA + CB)**2 % p; z3 = x1 * (DA - CB)**2 % p
        x2 = AA * BB % p; z2 = E * (AA + A24 * E) % p
    if swap: x2, x3, z2, z3 = x3, x2, z3, z2
    return (x2 * pow(z2, p - 2, p) % p).to_bytes(32, 'little')

h = bytes.fromhex
# RFC 7748, Section 5.2, first test vector
print(x25519(h('a546e36bf0527c9d3b16154b82465edd62144c0ac1fc5a18506a2244ba449ac4'),
             h('e6db6867583030db3594c1a424b15f7c726624ec26b3353b10a903a6d0ab1c4c')).hex()
      == 'c3da55379de9c6908e94ea4df28d084f32eccf03491c71f754b4075577a28552')
# RFC 7748, Section 6.1: Diffie-Hellman between Alice and Bob
nine = (9).to_bytes(32, 'little')
a_sk = h('77076d0a7318a57d3c16c17251b26645df4c2f87ebc0992ab177fba51db92c2a')
b_sk = h('5dab087e624a8a4b79e17f8b83800ee66f3bb1292618b6fd1c2f8b27ff88e0eb')
a_pk, b_pk = x25519(a_sk, nine), x25519(b_sk, nine)
print(a_pk.hex() == '8520f0098930a754748b7ddcb43ef75a0dbf3a0d26381af4eba4a98eaa9b4e6a',
      b_pk.hex() == 'de9edb7d7b7dc1b4d35b61c2ece435373f8343c85b78674dadfc7e146f882b4f')
s1, s2 = x25519(a_sk, b_pk), x25519(b_sk, a_pk)
print("shared secrets agree:", s1 == s2, s1.hex() == '4a5d9d5ba4ce2de1728e3bf480350f25e07e21c947d19e3376f09b3c1e161742')

In [ ]:
t0 = time.time()
E = pari.ellinit([0, 486662, 0, 1, 0], p25519)          # v^2 = u^3 + 486662 u^2 + u
N = int(pari.ellcard(E))
Nt = 2 * (p25519 + 1) - N                                 # the quadratic twist has 2p + 2 - N points
print(f"#E = 8 * l, l prime: {N % 8 == 0 and bool(pari.isprime(N // 8))};  l - 2^252 = {N // 8 - 2**252}")
print(f"#twist = 4 * l', l' prime: {Nt % 4 == 0 and bool(pari.isprime(Nt // 4))};  ({time.time() - t0:.1f}s)")
# a u-coordinate that is not on the curve lies on the twist; the ladder still works there
u = next(u for u in range(2, 100) if pari.kronecker((u**3 + 486662*u*u + u) % p25519, p25519) == -1)
print("u =", u, "is on the twist; X25519 still returns a value:", x25519(a_sk, u.to_bytes(32, 'little')).hex()[:16], "...")

Both groups have a huge prime factor: $\#E=8\ell$ with $\ell=2^{252}+27742317777372353535851937790883648493$, and the twist has $4\ell'$ points with $\ell'$ prime. The $u$-only ladder never looks at $v$, so a received $u$ that is not on the curve is silently a point on the twist; since the twist is also secure (*twist security*), an attacker gains nothing by sending it, and since the scalar is a multiple of $8$, points of small order are killed. This is why X25519 needs no point validation, unlike the Weierstrass-form curves of older standards, where a missing check on the received point enables the *invalid-curve attack* (the attacker sends points of small order on other curves $y^2=x^3+ax+b'$, which the addition formulas, not involving $b$, happily process, and recovers the secret key modulo many small primes, then by the Chinese remainder theorem). The mathematics of this chapter, the group law and its formulas, is exactly what such attacks exploit and what good designs defend against.